# Deadly Roads: Bangladesh 2000–2025, data pipeline

This notebook rebuilds every JSON file the website reads from the raw CSVs in `data/raw/`.
Run it from the repository root (`jupyter lab` in the repo folder, or `jupyter nbconvert --execute pipeline/pipeline.ipynb`).

What it does, in order:
1. National yearly series from eleven published sources, with deaths per 100,000 computed from World Bank population where a source gives deaths but no rate.
2. Highway corridors (OpenStreetMap trunk/primary alignments, routed) and 215 listed hazardous locations snapped to their corridor with a nearest-segment projection.
3. Section-ready tables for the Where, Who, When and Road-class chapters, derived from `breakdowns.csv` (2,471 rows transcribed from Road Safety Foundation and Jatri Kalyan Samity reports as reproduced in the press) and `monthly.csv`.
4. Programmatic checks: share tables must sum to 85–115%, every corridor must have geometry, every hazardous location must snap within a sensible distance.

Nothing is interpolated or modelled. Every raw row carries the URL it was transcribed from.

In [1]:
import os, sys, json
from pathlib import Path
REPO = Path.cwd() if (Path.cwd()/"data"/"raw").exists() else Path.cwd().parent
os.chdir(REPO); sys.argv = ["nb"]
print("repo:", REPO)

repo: /home/claude/work/repo


## 1 and 2. National series, corridors, hazardous locations, basemap layer

In [2]:
"""Build the JSON files the site reads from the raw CSVs. Run from the repo root: python pipeline/build_site_data.py"""
import json, math, re, sys
from pathlib import Path
import pandas as pd

ROOT = REPO
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data"
OUT.mkdir(parents=True, exist_ok=True)

def clean(v):
    if v is None or (isinstance(v, float) and math.isnan(v)): return None
    return v

def rows(df):
    return [{k: clean(v) for k, v in r.items()} for r in df.to_dict("records")]

# ---------- 1. national yearly ----------
nat = pd.read_csv(RAW / "national_yearly.csv")
pop = pd.read_csv(RAW / "population_national.csv").set_index("year")["population"]
# compute deaths per 100k where a source gives deaths but no rate (WHO rows keep their own published rate)
def rate(r):
    if pd.notna(r["deaths_per_100k"]): return float(r["deaths_per_100k"])
    if pd.notna(r["deaths"]) and r["year"] in pop.index: return round(float(r["deaths"]) / pop[r["year"]] * 1e5, 2)
    return None
nat["deaths_per_100k"] = nat.apply(rate, axis=1)
# short source labels used by the site
SHORT = {
    "Police/ARI-BUET (MAAP)": "Police/BRTA",
    "BRTA": "Police/BRTA",            # BRTA's own compilation of police cases, 2023-25
    "RHD (Mahmud et al 2013)": "RHD compilation",
    "BJKS": "BJKS", "RSF": "RSF", "NISCHA": "Nirapad Sarak Chai", "NCPSRR": "NCPSRR",
    "WHO GSRRS (modelled)": "WHO estimate",
    "WHO/World Bank SH.STA.TRAF.P5 (modelled rate)": "WHO/WB modelled rate",
    "BHIS (household survey)": "BHIS", "NHIS 2022-23 (household survey)": "BHIS",
}
nat["source_full"] = nat["source"]
nat["source"] = nat["source"].map(SHORT).fillna(nat["source"])
# police + BRTA overlap check: keep both rows if they conflict in the same year
dup = nat[nat.duplicated(["year", "source"], keep=False)]
if len(dup):
    print("WARN duplicate year/source rows:", dup[["year", "source", "deaths", "source_full"]].to_string())
    nat = nat.sort_values(["year", "source", "source_full"]).drop_duplicates(["year", "source"], keep="first")
nat = nat[nat.year >= 2000].sort_values(["source", "year"])
json.dump(rows(nat[["year", "source", "source_full", "accidents", "deaths", "injuries", "deaths_per_100k", "note", "citation_url"]]),
          open(OUT / "national.json", "w"), ensure_ascii=False)
print("national.json", len(nat), "rows; sources:", nat.source.unique().tolist())

# ---------- 2. corridors + blackspots ----------
poly = json.load(open(RAW / "roads" / "corridor_polylines.json"))
cor = pd.read_csv(RAW / "corridors.csv")
NAMES = {"N1": "Dhaka – Chattogram – Cox's Bazar – Teknaf", "N2": "Dhaka – Sylhet – Tamabil", "N3": "Dhaka – Mymensingh",
         "N4": "Dhaka – Tangail – Bangabandhu Bridge", "N405": "Elenga – Bangabandhu Bridge east approach", "N5": "Dhaka – Bogura – Rangpur – Banglabandha",
         "N6": "Dhaka – Natore – Rajshahi – Sona Masjid", "N7": "Daulatdia – Jashore – Khulna – Mongla", "N8": "Dhaka – Padma Bridge – Barishal", "N805": "Bhanga – Gopalganj – Mollahat (Dhaka–Khulna)"}
def hav(a, b):
    R = 6371000; la1, la2 = math.radians(a[1]), math.radians(b[1]); d1 = la2 - la1; d2 = math.radians(b[0] - a[0])
    s = math.sin(d1/2)**2 + math.cos(la1)*math.cos(la2)*math.sin(d2/2)**2; return 2*R*math.asin(math.sqrt(s))
corridors = {}
for ref, coords in poly.items():
    m = [0.0]
    for i in range(1, len(coords)): m.append(m[-1] + hav(coords[i-1], coords[i]))
    w = cor[cor.corridor == ref].sort_values("seq")
    corridors[ref] = {"ref": ref, "name": NAMES.get(ref, ref), "coords": [[round(x, 5), round(y, 5)] for x, y in coords], "total_m": round(m[-1]),
                      "official_route": clean(w.official_route.iloc[0]) if len(w) else None,
                      "wiki_length_km": clean(float(w.total_length_km.iloc[0])) if len(w) else None,
                      "waypoints": [{"place": r.place, "district": r.district, "km": float(r.km_from_start), "lng": float(r.lng), "lat": float(r.lat)} for r in w.itertuples()]}
json.dump(corridors, open(OUT / "corridors.json", "w"), ensure_ascii=False)
print("corridors.json", {k: (round(v["total_m"]/1000), len(v["coords"])) for k, v in corridors.items()})

bs = pd.read_csv(RAW / "blackspots.csv")
def precision(s):
    s = str(s).lower()
    if "upazila representative" in s or "point not located" in s or "bend not located" in s or "individual point not located" in s or "upazila polygon" in s: return "upazila"
    return "point"
bs["precision"] = bs.lat_lng_source.map(precision)
# route-metre of every spot on its corridor (nearest point on polyline)
def project(ref, p):
    coords = corridors[ref]["coords"]; best = (None, 1e18); m = 0.0
    kx = 111320*math.cos(math.radians(p[1])); ky = 111320
    for i in range(1, len(coords)):
        a, b = coords[i-1], coords[i]; L = hav(a, b)
        ax, ay = (a[0]-p[0])*kx, (a[1]-p[1])*ky; bx, by = (b[0]-p[0])*kx, (b[1]-p[1])*ky
        dx, dy = bx-ax, by-ay; LL = dx*dx+dy*dy; t = 0 if LL == 0 else max(0, min(1, -(ax*dx+ay*dy)/LL))
        d = math.hypot(ax+dx*t, ay+dy*t)
        if d < best[1]: best = (m + L*t, d)
        m += L
    return best
recs = []
for r in bs.itertuples():
    if r.highway not in corridors: continue
    m, d = project(r.highway, [float(r.lng), float(r.lat)])
    recs.append({"highway": r.highway, "place": r.name_place, "upazila": clean(r.upazila), "district": clean(r.district), "lat": float(r.lat), "lng": float(r.lng),
                 "route_m": round(m), "off_m": round(d), "precision": r.precision, "year": clean(int(r.year_listed)) if pd.notna(r.year_listed) else None,
                 "rank": clean(r.severity_or_rank), "source_url": clean(r.source_url), "note": clean(r.note), "located_by": clean(r.lat_lng_source)})
far = [x for x in recs if x["off_m"] > 8000]
print("blackspots.json", len(recs), "rows;", len(far), "more than 8 km off their corridor (kept, flagged)")
for x in far: x["flag"] = "far_from_corridor"
json.dump(recs, open(OUT / "blackspots.json", "w"), ensure_ascii=False)

# highway context layer: N and R roads, light simplification, split by class
g = json.load(open(RAW / "roads" / "bd-major-routes.geojson"))
feats = []
for f in g["features"]:
    ref = (f["properties"].get("ref") or "").strip()
    if not re.match(r"^[NR]\d+", ref): continue
    cls = "national" if ref.startswith("N") else "regional"
    c = [[round(x, 4), round(y, 4)] for x, y in f["geometry"]["coordinates"]]
    if len(c) < 2: continue
    feats.append({"type": "Feature", "properties": {"ref": ref.split(";")[0], "cls": cls}, "geometry": {"type": "LineString", "coordinates": c}})
json.dump({"type": "FeatureCollection", "features": feats}, open(OUT / "highways.geojson", "w"), separators=(",", ":"))
print("highways.geojson", len(feats), "features")

# ---------- 3. population + network ----------
pa = pd.read_csv(RAW / "population_admin.csv")
json.dump(rows(pa[["level", "name", "division", "pop_2011", "pop_2022", "area_km2"]]), open(OUT / "population_admin.json", "w"), ensure_ascii=False)
json.dump(rows(pd.read_csv(RAW / "population_national.csv")[["year", "population"]]), open(OUT / "population_national.json", "w"))
print("done; now run build_sections.py")


national.json 89 rows; sources: ['BHIS', 'BJKS', 'NCPSRR', 'Nirapad Sarak Chai', 'Police/BRTA', 'RHD compilation', 'RSF', 'WHO estimate', 'WHO/WB modelled rate']
corridors.json {'N1': (439, 528), 'N2': (280, 295), 'N3': (109, 89), 'N4': (149, 194), 'N405': (16, 18), 'N5': (511, 500), 'N6': (227, 247), 'N7': (253, 246), 'N8': (198, 223), 'N805': (122, 135)}
blackspots.json 215 rows; 20 more than 8 km off their corridor (kept, flagged)


highways.geojson 3609 features
done; now run build_sections.py


## 3. Section tables (where, who, when, roads)

In [3]:
"""Second pipeline step: derive section-ready JSON (where, who, when, roads) from breakdowns.csv and monthly.csv.
Every derived number carries the list of periods it was computed from, so the page can say exactly what it is showing."""
import json, math, re
from pathlib import Path
import pandas as pd

ROOT = REPO
RAW = ROOT / "data" / "raw"; OUT = ROOT / "data"
b = pd.read_csv(RAW / "breakdowns.csv")
mo = pd.read_csv(RAW / "monthly.csv")
mo = mo[mo.year.astype(str).str.isdigit()].copy(); mo["year"] = mo.year.astype(int); mo["month"] = mo.month.astype(int)
b["value"] = pd.to_numeric(b["value"], errors="coerce")
DIVS = ["Dhaka", "Chattogram", "Rajshahi", "Khulna", "Barishal", "Sylhet", "Rangpur", "Mymensingh"]
MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

def period_label(p):
    m = re.match(r"^(\d{4})-(\d{2})$", str(p))
    if m: return f"{MONTHS[int(m.group(2))-1]} {m.group(1)}"
    m = re.match(r"^eid_(fitr|azha)_(\d{4})$", str(p))
    if m: return f"Eid-ul-{'Fitr' if m.group(1)=='fitr' else 'Azha'} {m.group(2)}"
    m = re.match(r"^(\d{4})-(\d{2})_to_(\d{4})-(\d{2})$", str(p))
    if m: return f"{MONTHS[int(m.group(2))-1]} {m.group(1)} to {MONTHS[int(m.group(4))-1]} {m.group(3)}"
    return str(p).replace("_", " ")

def cite(df):
    return sorted(set(df.citation_url.dropna()))

# ---------------- WHERE: division shares ----------------
d = b[(b.dimension == "division") & (b.source == "RSF")]
periods = []
for p, g in d.groupby("period"):
    sh = g[g.measure == "share_pct"].copy()
    sh["is_death"] = sh.note.fillna("").str.lower().str.contains("death")
    # prefer accident shares; if a period only has death shares use those
    pref = sh[~sh.is_death] if (~sh.is_death).sum() >= 7 else sh[sh.is_death]
    pref = pref.drop_duplicates(["category"], keep="first")
    shares = {r.category: float(r.value) for r in pref.itertuples() if r.category in DIVS}
    if len(shares) < 7 or not (85 <= sum(shares.values()) <= 115): continue
    basis_from_shares = "deaths" if (len(pref) and pref.is_death.iloc[0]) else "accidents"
    tot = b[(b.source == "RSF") & (b.period == p) & (b.dimension == "total") & (b.category == "all")]
    totals = {r.measure: float(r.value) for r in tot.itertuples()}
    # monthly.csv has month totals too
    m = re.match(r"^(\d{4})-(\d{2})$", str(p))
    if m:
        mm = mo[(mo.source == "RSF") & (mo.year == int(m.group(1))) & (mo.month == int(m.group(2)))]
        if len(mm):
            for k in ["accidents", "deaths", "injuries"]:
                if pd.notna(mm.iloc[0][k]): totals.setdefault(k, float(mm.iloc[0][k]))
    # the share is a share of accidents unless the note says deaths; RSF press tables are accident shares
    basis = basis_from_shares
    cnt = {r.category: float(r.value) for r in g[g.measure == basis].itertuples() if r.category in DIVS}
    periods.append({"period": p, "label": period_label(p), "shares": shares, "basis": basis, "totals": totals, "counts_published": cnt, "sources": cite(g)})
periods.sort(key=lambda x: x["period"])
# pooled: weighted by period accident total when known, else equal weight
pool = {k: 0.0 for k in DIVS}; wsum = 0.0
for pr in periods:
    w = pr["totals"].get("accidents", 1.0) if pr["basis"] == "accidents" else pr["totals"].get("deaths", 1.0)
    for k in DIVS: pool[k] += pr["shares"].get(k, 0) * w
    wsum += w
pooled = {k: round(v / wsum, 2) for k, v in pool.items()}
# annual highest/lowest rows
ann = d[(d.period == "annual")]
annual_rows = [{"year": int(r.year) if str(r.year).isdigit() else r.year, "division": r.category, "measure": r.measure, "value": float(r.value), "url": r.citation_url} for r in ann.itertuples() if r.category in DIVS]
json.dump({"divisions": DIVS, "periods": periods, "pooled": pooled, "pooled_n": len(periods), "annual_extremes": annual_rows},
          open(OUT / "where.json", "w"), ensure_ascii=False, indent=0)
print("where.json periods:", [p["period"] for p in periods])

# ---------------- WHO ----------------
rsf = b[(b.source == "RSF") & (b.period == "annual")]
def series(dim, cats, measure="deaths"):
    out = {}
    for y, g in rsf[(rsf.dimension == dim) & (rsf.measure == measure)].groupby("year"):
        if not str(y).isdigit(): continue
        row = {}
        for c in cats:
            v = g[g.category == c].value
            if len(v): row[c] = float(v.iloc[0])
        if row: out[int(y)] = row
    return out
victims = series("victim", ["women", "children", "pedestrian", "drivers_helpers", "motorcycle", "motorcycle_all", "students", "working_age_18_65"])
veh_counts = series("vehicle_type", ["motorcycle", "three_wheeler", "local_three_wheeler", "truck_pickup_goods", "bus", "car_microbus", "local_vehicle", "bicycle_rickshaw", "bus_truck_car_combined"])
veh_share = series("vehicle_type", ["motorcycle", "three_wheeler", "local_three_wheeler", "truck_pickup_goods", "bus", "car_microbus", "local_vehicle", "bicycle_rickshaw", "bus_truck_car_combined"], "share_pct")
moto_acc = series("vehicle_type", ["motorcycle"], "accidents")
totals = {int(r.year): float(r.value) for r in rsf[(rsf.dimension == "total") & (rsf.measure == "deaths")].itertuples() if str(r.year).isdigit()}
tot_acc = {int(r.year): float(r.value) for r in rsf[(rsf.dimension == "total") & (rsf.measure == "accidents")].itertuples() if str(r.year).isdigit()}
bj = b[(b.source == "BJKS") & (b.period == "annual")]
bj_moto = {int(r.year): float(r.value) for r in bj[(bj.dimension == "vehicle_type") & (bj.category == "motorcycle") & (bj.measure == "deaths")].itertuples() if str(r.year).isdigit()}
bj_victims = {}
for y, g in bj[(bj.dimension == "victim") & (bj.measure == "deaths")].groupby("year"):
    if str(y).isdigit(): bj_victims[int(y)] = {r.category: float(r.value) for r in g.itertuples()}
# collision types: pooled RSF monthly accident counts + BJKS annual shares
col = b[(b.dimension == "collision_type")]
rsf_col = col[(col.source == "RSF") & (col.measure == "accidents") & col.period.str.match(r"^\d{4}-\d{2}$")]
pooled_col = rsf_col.groupby("category").value.sum().to_dict(); col_months = sorted(rsf_col.period.unique().tolist())
bj_col = {}
for y, g in col[(col.source == "BJKS") & (col.period == "annual") & (col.measure == "share_pct")].groupby("year"):
    if str(y).isdigit(): bj_col[int(y)] = {r.category: float(r.value) for r in g.itertuples()}
who = {"rsf_totals_deaths": totals, "rsf_totals_accidents": tot_acc, "victims": victims, "veh_counts": veh_counts, "veh_share": veh_share, "moto_accidents": moto_acc,
       "bjks_moto_deaths": bj_moto, "bjks_victims": bj_victims, "collision_pooled_rsf": pooled_col, "collision_months": col_months, "collision_bjks": bj_col,
       "sources": cite(pd.concat([rsf[rsf.dimension.isin(["victim", "vehicle_type", "total"])], bj[bj.dimension.isin(["victim", "vehicle_type"])], col]))}
json.dump(who, open(OUT / "who.json", "w"), ensure_ascii=False, indent=0)
print("who.json victims years:", sorted(victims), "veh_counts:", sorted(veh_counts))

# ---------------- WHEN ----------------
BANDS = {"dawn": [4, 6], "early_morning": [4, 6], "morning": [6, 12], "noon": [12, 15], "afternoon": [15, 18], "late_afternoon": [15, 18], "evening": [18, 20], "dusk": [18, 20], "night": [20, 28]}
tod = b[(b.dimension == "time_of_day") & (b.source == "RSF") & (b.measure == "share_pct")]
tod_periods = []
for p, g in tod.groupby("period"):
    sh = {r.category: float(r.value) for r in g.itertuples()}
    cov = sum(sh.values())
    if len(sh) >= 5 and 90 <= cov <= 110:
        tod_periods.append({"period": p, "label": period_label(p), "shares": sh, "coverage_pct": round(cov, 1), "sources": cite(g)})
tod_periods.sort(key=lambda x: x["period"])
# pooled time of day: simple mean of full-coverage periods (no totals for Eid periods vs months; say so)
keys = sorted({k for pr in tod_periods for k in pr["shares"]})
canon = {}
for pr in tod_periods:
    for k, v in pr["shares"].items():
        ck = {"early_morning": "dawn", "dusk": "evening", "late_afternoon": "afternoon"}.get(k, k)
        canon.setdefault(ck, []).append(v)
tod_pooled = {k: sum(v) / len(v) for k, v in canon.items()}
_tot = sum(tod_pooled.values()) or 1
tod_pooled = {k: round(v / _tot * 100, 2) for k, v in tod_pooled.items()}  # renormalised to 100 (published tables round to 99-101)
cum = b[(b.dimension == "time_of_day") & (b.period == "2020_to_2026-07") & (b.measure == "deaths")]
cum_rows = {r.category: float(r.value) for r in cum.itertuples()}
cum_note = " ".join(cum.note.dropna().astype(str).unique().tolist())
# monthly series RSF 2020-2026
mrows = mo[mo.source == "RSF"].copy()
mrows = mrows[mrows.year.astype(str).str.isdigit()]
monthly = [{"year": int(r.year), "month": int(r.month), "accidents": None if pd.isna(r.accidents) else float(r.accidents), "deaths": None if pd.isna(r.deaths) else float(r.deaths), "injuries": None if pd.isna(r.injuries) else float(r.injuries), "url": r.citation_url} for r in mrows.itertuples()]
# Eid windows
eid = b[b.period.str.startswith("eid") & (b.dimension == "total") & (b.source == "RSF")]
eids = {}
for p, g in eid.groupby("period"):
    rec = {r.measure: float(r.value) for r in g.itertuples()}
    rec["label"] = period_label(p); rec["year"] = int(p[-4:]); rec["kind"] = "Fitr" if "fitr" in p else "Azha"; rec["sources"] = cite(g)
    eids[p] = rec
json.dump({"bands": BANDS, "tod_periods": tod_periods, "tod_pooled": tod_pooled, "tod_pooled_n": len(tod_periods), "cumulative_deaths_2020_2026": cum_rows, "cumulative_note": cum_note,
           "monthly": monthly, "eid": eids, "sources": cite(pd.concat([tod, eid]))}, open(OUT / "when.json", "w"), ensure_ascii=False, indent=0)
print("when.json tod periods:", [p["period"] for p in tod_periods], "monthly rows:", len(monthly), "eid:", sorted(eids))

# ---------------- ROADS ----------------
rc = b[b.dimension == "road_class"]
CLASS_MAP = {"national_highway": "National highway", "regional_highway": "Regional highway", "regional_road": "Regional highway", "feeder_road": "Feeder / rural road", "rural_road": "Feeder / rural road",
             "city_road": "City road", "dhaka_city": "City road", "chattogram_city": "City road", "other": "Other", "railway_crossing": "Other"}
shares = []
for (s, p), g in rc[rc.measure == "share_pct"].groupby(["source", "period"]):
    row = {}
    gg = g.groupby("category").value.mean()
    for cat, v in gg.items():
        k = CLASS_MAP.get(cat, cat); row[k] = row.get(k, 0) + float(v)
    if not (85 <= sum(row.values()) <= 115): print("  skip road share", s, p, round(sum(row.values()),1)); continue
    yr = g.year.iloc[0]
    shares.append({"source": s, "period": p, "label": period_label(p) if p != "annual" else str(yr), "year": str(yr), "shares": row, "sources": cite(g)})
shares.sort(key=lambda x: (x["source"], x["period"]))
monthly_rc = []
for p, g in rc[(rc.source == "RSF") & (rc.measure == "accidents") & rc.period.str.match(r"^\d{4}-\d{2}$")].groupby("period"):
    row = {}
    gg = g.groupby("category").value.mean()
    for cat, v in gg.items():
        k = CLASS_MAP.get(cat, cat); row[k] = row.get(k, 0) + float(v)
    monthly_rc.append({"period": p, "label": period_label(p), "counts": row, "sources": cite(g)})
monthly_rc.sort(key=lambda x: x["period"])
rn = pd.read_csv(RAW / "road_network.csv")
json.dump({"shares": shares, "monthly_counts": monthly_rc, "network": [{k: (None if (isinstance(v, float) and math.isnan(v)) else v) for k, v in r.items()} for r in rn.to_dict("records")]},
          open(OUT / "roads.json", "w"), ensure_ascii=False, indent=0)
print("roads.json shares:", [(s["source"], s["label"]) for s in shares], "monthly:", len(monthly_rc))

# ---------------- METHOD: citation index ----------------
urls = sorted(set(b.citation_url.dropna()) | set(mo.citation_url.dropna()) | set(pd.read_csv(RAW / "national_yearly.csv").citation_url.dropna()) | set(pd.read_csv(RAW / "blackspots.csv").source_url.dropna()))
json.dump({"n_citations": len(urls), "n_breakdown_rows": int(len(b)), "n_monthly_rows": int(len(mo))}, open(OUT / "method.json", "w"))
print("citations:", len(urls))


where.json periods: ['2022-06', '2022-07', '2022-08', '2025-08', '2025-09', '2025-10', '2025-11', '2026-06', 'eid_fitr_2025']
who.json victims years: [2020, 2021, 2022, 2023, 2024, 2025] veh_counts: [2020, 2021, 2022, 2023, 2024, 2025]
when.json tod periods: ['2024-10', '2025-01', '2025-07', '2026-06', 'eid_fitr_2025'] monthly rows: 44 eid: ['eid_azha_2020', 'eid_azha_2021', 'eid_azha_2022', 'eid_azha_2023', 'eid_azha_2024', 'eid_azha_2025', 'eid_azha_2026', 'eid_fitr_2023', 'eid_fitr_2024', 'eid_fitr_2025']
  skip road share RSF 2020-08 37.4
  skip road share RSF 2021-01 63.4
  skip road share RSF 2022-11 41.5
  skip road share RSF 2025-03 82.5
  skip road share RSF 2026-03 76.0


roads.json shares: [('BJKS', '2014 to 2024-09'), ('BJKS', 'Jan 2023'), ('BJKS', 'Jan 2024'), ('BJKS', 'May 2025'), ('BJKS', 'Jun 2026'), ('BJKS', '2019'), ('BJKS', 'Eid-ul-Azha 2025'), ('BJKS', 'Eid-ul-Fitr 2025'), ('RSF', 'Jan 2019 to Jun 2024'), ('RSF', '2020 to 2026-07'), ('RSF', 'May 2022'), ('RSF', 'Feb 2023'), ('RSF', 'Jun 2023'), ('RSF', 'Jul 2023'), ('RSF', 'Sep 2023'), ('RSF', 'Nov 2023'), ('RSF', 'Jan 2024 to Sep 2024'), ('RSF', 'Feb 2024'), ('RSF', 'Oct 2024'), ('RSF', 'Jan 2025'), ('RSF', 'Feb 2025'), ('RSF', 'Sep 2025'), ('RSF', 'Jan 2026'), ('RSF', 'Jun 2026'), ('RSF', 'Jul 2026'), ('RSF', 'Aug 2026'), ('RSF', 'Sep 2026'), ('RSF', '2023'), ('RSF', 'Eid-ul-Azha 2025'), ('RSF', 'Eid-ul-Azha 2026'), ('RSF', 'Eid-ul-Fitr 2025'), ('RSF', 'unknown-05')] monthly: 28
citations: 170


## 4. Checks

In [4]:
import json, pandas as pd
D = REPO/"data"
nat = json.load(open(D/"national.json")); assert len(nat) > 80, "national series too short"
cor = json.load(open(D/"corridors.json")); assert all(v["total_m"] > 10000 and len(v["coords"]) > 10 for v in cor.values())
spots = json.load(open(D/"blackspots.json")); far = [s for s in spots if s["off_m"] > 8000]
print(f"{len(spots)} hazardous locations, {len(far)} more than 8 km off their corridor (flagged, kept)")
where = json.load(open(D/"where.json")); assert 95 <= sum(where["pooled"].values()) <= 105, where["pooled"]
for p in where["periods"]: assert 85 <= sum(p["shares"].values()) <= 115, p["period"]
roads = json.load(open(D/"roads.json"))
for s in roads["shares"]: assert 85 <= sum(s["shares"].values()) <= 115, (s["source"], s["period"])
when = json.load(open(D/"when.json")); assert 99 <= sum(when["tod_pooled"].values()) <= 101
who = json.load(open(D/"who.json")); assert all(y in who["rsf_totals_deaths"] for y in ["2020","2021","2022","2023","2024","2025"])
for f in ["national_yearly.csv","breakdowns.csv","monthly.csv"]:
    df = pd.read_csv(D/"raw"/f); assert df["citation_url"].notna().all(), f
bs = pd.read_csv(D/"raw"/"blackspots.csv"); assert bs.source_url.notna().all() and bs.lat.notna().all()
print("all checks passed")

215 hazardous locations, 20 more than 8 km off their corridor (flagged, kept)
all checks passed
